# 0825_lsw_006_sliding_window_drift

로드맵 Phase 2(Distribution Drift 대응)에서 아직 안 한 부분: 003/004/005는 "고정된 60:20:20
분할 하나"로만 성능을 봤다. 이번엔 **같은 크기(40%)의 학습 윈도우를 뒤로 밀면서(sliding window)
다시 학습**했을 때, 검증 구간 성능이 뒤로 갈수록 달라지는지 확인한다.

- Step 1: Train 0~40% → Validation 40~60%
- Step 2: Train 20~60% → Validation 60~80%
- Step 3: Train 40~80% → Validation 80~100%

매 step마다 **그 step의 Train으로 새로 학습하고, 그 step의 Validation으로 임계값을 새로 고른
뒤, 같은 Validation에서 평가**한다(003~005처럼 별도 Test로 넘기지 않고, "이 구간에서 학습하면
이 정도 성능이 나온다"를 바로 본다 — 사용자가 제안한 단순 sliding-window 진단).

Train 크기·재학습 방식은 매 step마다 동일하게 유지하므로(같은 40%, 같은 학습법), **step 간
성능 차이가 있다면 그건 재학습 부족 때문이 아니라 데이터 자체가 시간에 따라 더 어려워지고
있다는(=drift) 증거**로 해석할 수 있다.


## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_lsw_006_sliding_window_drift"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}
MIN_TRAIN_POS = 5

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)


experiment: 0825_lsw_006_sliding_window_drift


## 2. 데이터 로딩·전처리 (003/004/005와 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]
timestamps = clean_df[TIME_COLUMN]


def cut_at(fraction):
    sizes = timestamps.value_counts(sort=False).sort_index()
    cum = sizes.cumsum().to_numpy()
    idx = int(np.searchsorted(cum, len(clean_df) * fraction, side="left"))
    idx = min(idx, len(sizes) - 1)
    return sizes.index[idx]


cut_points = {f: cut_at(f) for f in [0.20, 0.40, 0.60, 0.80]}
print("cut points:", cut_points)
print("data range:", timestamps.min(), "~", timestamps.max())


cut points: {0.2: Timestamp('1970-08-02 17:57:15+0000', tz='UTC'), 0.4: Timestamp('1970-08-25 14:06:02+0000', tz='UTC'), 0.6: Timestamp('1970-09-28 05:48:26+0000', tz='UTC'), 0.8: Timestamp('1970-10-13 13:14:26+0000', tz='UTC')}
data range: 1970-06-23 03:58:55+00:00 ~ 1970-11-02 14:21:28+00:00


## 3. 평가 함수 (003/004/005와 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "n": len(y_true),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()


def build_model():
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
    )


## 4. Sliding window 3-step 정의

같은 폭(Train 40% / Val 20%)을 20%씩 뒤로 밀며 3개 step을 만든다.

In [4]:
windows = [
    {"step": 1, "train_hi": cut_points[0.40], "val_lo": cut_points[0.40], "val_hi": cut_points[0.60]},
    {"step": 2, "train_lo": cut_points[0.20], "train_hi": cut_points[0.60], "val_lo": cut_points[0.60], "val_hi": cut_points[0.80]},
    {"step": 3, "train_lo": cut_points[0.40], "train_hi": cut_points[0.80], "val_lo": cut_points[0.80], "val_hi": None},
]

for w in windows:
    train_lo = w.get("train_lo")
    train_mask = (timestamps > train_lo) if train_lo is not None else pd.Series(True, index=clean_df.index)
    train_mask &= timestamps <= w["train_hi"]
    val_hi = w.get("val_hi")
    val_mask = (timestamps > w["val_lo"])
    if val_hi is not None:
        val_mask &= timestamps <= val_hi
    print(f"step {w['step']}: train {int(train_mask.sum())} rows, val {int(val_mask.sum())} rows")


step 1: train 156800 rows, val 78422 rows
step 2: train 156816 rows, val 78374 rows
step 3: train 156796 rows, val 78396 rows


## 5. 검사유형×step 학습·평가

In [5]:
sliding_results = []

for w in windows:
    step = w["step"]
    train_lo = w.get("train_lo")
    train_hi = w["train_hi"]
    val_lo = w["val_lo"]
    val_hi = w.get("val_hi")

    train_mask = (timestamps > train_lo) if train_lo is not None else pd.Series(True, index=clean_df.index)
    train_mask &= timestamps <= train_hi
    val_mask = timestamps > val_lo
    if val_hi is not None:
        val_mask &= timestamps <= val_hi

    for inspection_type in sorted(clean_df["inspection_type"].unique()):
        type_mask = clean_df["inspection_type"] == inspection_type
        train_df = clean_df.loc[train_mask & type_mask]
        val_df = clean_df.loc[val_mask & type_mask]

        n_train_pos = int((train_df[TARGET] == 1).sum())
        if n_train_pos < MIN_TRAIN_POS or len(val_df) == 0:
            sliding_results.append(
                {"step": step, "inspection_type": inspection_type, "insufficient_data": True,
                 "n_train_pos": n_train_pos, "n_val": len(val_df)}
            )
            continue

        feature_columns = get_non_constant_columns(feature_columns_all, train_df)
        model = build_model()
        model.fit(train_df[feature_columns], train_df[TARGET])

        val_proba = model.predict_proba(val_df[feature_columns])[:, 1]
        threshold = select_threshold(val_df[TARGET], val_proba)

        result = evaluate_at_threshold(val_df[TARGET], val_proba, threshold)
        result["step"] = step
        result["inspection_type"] = inspection_type
        result["insufficient_data"] = False
        result["n_train_pos"] = n_train_pos
        sliding_results.append(result)

sliding_df = pd.DataFrame(sliding_results).set_index(["inspection_type", "step"]).sort_index()
sliding_df


threshold      n     tn     fp  fn   tp  slip_rate  \
inspection_type step                                                          
0               1     4.529815e-06  13409   6456   6932   0   21   0.000000   
                2     4.212050e-06  23492  10685  12799   0    8   0.000000   
                3     3.604244e-07  16784   1117  15534   1  132   0.007519   
1               1     6.899076e-06   7097   1440   5399   2  256   0.007752   
                2     3.439553e-05  10494   2529   7726   2  237   0.008368   
                3     1.000968e-05  11112   1151   9177   7  777   0.008929   
2               1     4.816745e-10  22317    142  22096   0   79   0.000000   
                2     1.574087e-06  15897    700  15165   0   32   0.000000   
                3     4.504793e-07  19153    310  18140   7  696   0.009957   
3               1     4.735841e-06  34547   5105  29381   0   61   0.000000   
                2     4.711129e-06  27328   7538  19766   0   24   0.000000   
                3     3.250585e-07  30591    630  29358   6  597   0.009950   
4               1     1.100488e-04   1052    800    247   0    5   0.000000   
                2     4.467599e-06   1163      6   1097   0   60   0.000000   
                3     3.445915e-05    756    170    560   0   26   0.000000   

                      volume_reduction    pr_auc  total_cost_1:10  \
inspection_type step                                                
0               1             0.482223  0.015884             6932   
                2             0.454991  0.000715            12799   
                3             0.067083  0.044287            15544   
1               1             0.210557  0.192772             5419   
                2             0.246611  0.565164             7746   
                3             0.111445  0.581261             9247   
2               1             0.006385  0.279031            22096   
                2             0.044122  0.122925            15165   
                3             0.016802  0.349967            18210   
3               1             0.148031  0.085529            29381   
                2             0.276077  0.026429            19766   
                3             0.021008  0.244641            29418   
4               1             0.764088  0.047724              247   
                2             0.005440  0.062422             1097   
                3             0.232877  0.233893              560   

                      total_cost_1:100  insufficient_data  n_train_pos  
inspection_type step                                                    
0               1                 6932              False           36  
                2                12799              False           40  
                3                15634              False           29  
1               1                 5599              False          297  
                2                 7926              False          342  
                3                 9877              False          497  
2               1                22096              False          500  
                2                15165              False          289  
                3                18840              False          111  
3               1                29381              False          559  
                2                19766              False          359  
                3                29958              False           85  
4               1                  247              False            8  
                2                 1097              False            9  
                3                  560              False           65

## 6. Step별 추세 (검사유형별)

In [6]:
trend_columns = ["n_train_pos", "n", "threshold", "tn", "fp", "fn", "tp", "slip_rate", "volume_reduction", "pr_auc", "total_cost_1:10", "total_cost_1:100"]
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    print(f"=== type {inspection_type} ===")
    display(sliding_df.loc[inspection_type][[c for c in trend_columns if c in sliding_df.columns]])


=== type 0 ===


,n_train_pos,n,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,total_cost_1:10,total_cost_1:100
step,,,,,,,,,,,,
1,36,13409,4.529815e-06,6456,6932,0,21,0.000000,0.482223,0.015884,6932,6932
2,40,23492,4.212050e-06,10685,12799,0,8,0.000000,0.454991,0.000715,12799,12799
3,29,16784,3.604244e-07,1117,15534,1,132,0.007519,0.067083,0.044287,15544,15634


=== type 1 ===


,n_train_pos,n,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,total_cost_1:10,total_cost_1:100
step,,,,,,,,,,,,
1,297,7097,0.000007,1440,5399,2,256,0.007752,0.210557,0.192772,5419,5599
2,342,10494,0.000034,2529,7726,2,237,0.008368,0.246611,0.565164,7746,7926
3,497,11112,0.000010,1151,9177,7,777,0.008929,0.111445,0.581261,9247,9877


=== type 2 ===


,n_train_pos,n,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,total_cost_1:10,total_cost_1:100
step,,,,,,,,,,,,
1,500,22317,4.816745e-10,142,22096,0,79,0.000000,0.006385,0.279031,22096,22096
2,289,15897,1.574087e-06,700,15165,0,32,0.000000,0.044122,0.122925,15165,15165
3,111,19153,4.504793e-07,310,18140,7,696,0.009957,0.016802,0.349967,18210,18840


=== type 3 ===


,n_train_pos,n,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,total_cost_1:10,total_cost_1:100
step,,,,,,,,,,,,
1,559,34547,4.735841e-06,5105,29381,0,61,0.00000,0.148031,0.085529,29381,29381
2,359,27328,4.711129e-06,7538,19766,0,24,0.00000,0.276077,0.026429,19766,19766
3,85,30591,3.250585e-07,630,29358,6,597,0.00995,0.021008,0.244641,29418,29958


=== type 4 ===


,n_train_pos,n,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,total_cost_1:10,total_cost_1:100
step,,,,,,,,,,,,
1,8,1052,0.000110,800,247,0,5,0.0,0.764088,0.047724,247,247
2,9,1163,0.000004,6,1097,0,60,0.0,0.005440,0.062422,1097,1097
3,65,756,0.000034,170,560,0,26,0.0,0.232877,0.233893,560,560


## 7. 결론 및 다음 단계

### Volume Reduction 추세 (step1 → step2 → step3, 매 step 새로 학습한 결과)

| type | step1 | step2 | step3 | 패턴 |
|---|---:|---:|---:|---|
| 0 | 48.2% | 45.5% | **6.7%** | step1→2는 유지, step3에서 급락(약 1/7) |
| 1 | 21.1% | 24.7% | **11.1%** | step2까지 개선되다 step3에서 절반 이하로 급락 |
| 2 | 0.6% | 4.4% | 1.7% | 전 구간 낮음(2~9%), 뚜렷한 방향성 없이 노이즈 수준 |
| 3 | 14.8% | 27.6% | **2.1%** | step2까지 거의 2배 개선되다 step3에서 거의 0으로 붕괴 |
| 4 | 76.4% | 0.5% | 23.3% | 표본 8~65건 수준이라 추세라기보다 노이즈 |

**핵심 발견: type0/1/3은 매 step마다 그 시점 기준 최신 40%로 새로 학습했는데도 step3(80~100%
구간)에서 Volume Reduction이 뚜렷하게 붕괴한다.** Train 크기·재학습 방식이 동일하므로 이건
"모델이 낡아서"가 아니라 **데이터 자체가 마지막 구간에서 더 어려워졌다는 증거**다 — 003/004에서
확인한 "train 0.72% → test 2.64%" 불량률 급등, notes.md의 "판정 기준이 시간이 갈수록
엄격해짐(0→1)" 발견과 정확히 같은 방향이다. 이 구간에서는 Slip Rate ≤1% 제약을 만족하려는
임계값이 극단적으로 낮아지면서(threshold가 1e-7 수준까지 내려감) FP가 폭증하고 Volume Reduction이
무너진다.

**type1은 PR-AUC는 오히려 계속 좋아지는데(0.19→0.57→0.58) Volume Reduction은 나빠진다** — 반복
관찰된 "PR-AUC 개선 ≠ 운영 지표 개선" 패턴이 여기서도 나타난다. 랭킹 능력 자체는 좋아지지만, Slip
Rate 제약이 걸리는 극단적 저확률 구간의 캘리브레이션이 흔들리는 것으로 보인다.

**type2는 전 구간에서 Volume Reduction이 낮다(2~9%)** — 003/004에서부터 이미 알려진 약점(모순
라벨 28개, 판정기준 변화가 가장 심한 유형)과 일관된다. 뚜렷한 시간 추세보다는 "원래 어려운
유형"이라는 해석에 더 가깝다.

**type4는 표본이 너무 적어(step별 양성 8/9/65건) 이 실험으로는 추세를 판단할 수 없다** — 004/005에서
반복 확인된 한계와 동일.

### Phase 2 로드맵 관점에서의 의미

- 003/004/005는 "고정된 60:20:20 하나"만 봤지만, 이번 sliding-window 진단으로 **재학습을 계속
  해도(같은 40% 크기로 매번 새로 학습) 마지막 구간에서는 성능이 떨어진다**는 게 확인됐다. 즉
  "주기적 재학습"만으로는 이 데이터의 drift를 완전히 상쇄할 수 없다 — 005의 결론(recency
  weighting도 만능이 아님)과 같은 방향이다.
- 이 결과는 이전에 사용자에게 설명했던 "PSI 기반 트리거로는 concept drift를 못 잡는다"는 논리를
  데이터로 보강한다: 여기서 문제가 되는 건 데이터 볼륨/분포가 아니라 **Slip Rate 제약 하에서
  임계값이 감당할 수 있는 여유(margin)가 시간이 갈수록 줄어드는 것**이다.
- 정적 데이터셋이라 "그 다음엔 어떻게 되는지" 더 미래를 볼 수는 없다 — 이 132일 구간 안에서
  관측된 패턴이 이후에도 계속될지는 검증 불가능하다는 한계를 명시한다.

### 다음 단계

1. 이 노트북 결과를 `docs/experiments/0825_lsw_006_sliding_window_drift.md`로 옮기고
   `docs/experiments/index.md`, `handoff.md`에 반영한다.
2. 이상치 탐지 재구현(라벨 무관 극단값 기준) — 여전히 미착수.
3. Phase 4(비지도 이상탐지)로 이동 — feature selection/drift 대응/라벨 강건 학습 모두 결론이
   났고, 004의 검사유형별 최적 조합표가 여전히 최종 baseline이다.


## 8. 추가 검증 — "재학습 데이터가 적어서 손해"와 "최신성" 효과를 분리

사용자 지적: 5절의 step들은 Train 크기(40%)는 고정했지만, **원래 baseline(Train 60%)보다는
어차피 다 작다.** 그리고 step2/step3가 나빠 보이는 게 "최근이라서"가 아니라 "그 시점까지의
데이터가 우연히 적어서"일 수도 있다는 의심을 풀어야 한다. 그래서 **Val을 하나로 고정(60~80%,
즉 step2와 동일)**하고, Train을 4가지 조합으로 바꿔가며 **크기(size)와 최신성(gap, Train
끝~Val 시작 사이 간격)을 독립적으로** 바꿔본다.

| 이름 | Train 구간 | 크기 | Val과의 gap |
|---|---|---|---|
| size40_gap0 | 20~60% | 40% (=step2와 동일) | 0%p (바로 이어짐) |
| size40_gap20 | 0~40% | 40% | 20%p (더 예전 데이터) |
| size20_gap0 | 40~60% | 20% | 0%p (바로 이어지지만 적은 양) |
| size20_gap20 | 20~40% | 20% | 20%p (적고 예전) |

**size40_gap0 vs size40_gap20**: 크기는 같은데 최신성만 다르다 → 최신성 자체의 효과.
**size40_gap20 vs size20_gap20**: gap은 같은데 크기만 다르다(사용자가 요청한 비교) → 데이터
양 자체의 효과. 여기서 **size가 더 큰데도 오히려 지면**, "학습 데이터가 적어서 손해"라는
설명으로는 부족하고 최신성/drift 효과가 진짜로 있다는 뜻이 된다.


In [7]:
val_lo, val_hi = cut_points[0.60], cut_points[0.80]

variants = [
    {"name": "size40_gap0", "train_lo": cut_points[0.20], "train_hi": cut_points[0.60]},
    {"name": "size40_gap20", "train_lo": None, "train_hi": cut_points[0.40]},
    {"name": "size20_gap0", "train_lo": cut_points[0.40], "train_hi": cut_points[0.60]},
    {"name": "size20_gap20", "train_lo": cut_points[0.20], "train_hi": cut_points[0.40]},
]

volume_gap_results = []
for variant in variants:
    train_lo = variant["train_lo"]
    train_hi = variant["train_hi"]
    train_mask_v = (timestamps > train_lo) if train_lo is not None else pd.Series(True, index=clean_df.index)
    train_mask_v &= timestamps <= train_hi
    val_mask_v = (timestamps > val_lo) & (timestamps <= val_hi)

    for inspection_type in sorted(clean_df["inspection_type"].unique()):
        type_mask = clean_df["inspection_type"] == inspection_type
        train_df = clean_df.loc[train_mask_v & type_mask]
        val_df = clean_df.loc[val_mask_v & type_mask]

        n_train_pos = int((train_df[TARGET] == 1).sum())
        if n_train_pos < MIN_TRAIN_POS:
            volume_gap_results.append(
                {"variant": variant["name"], "inspection_type": inspection_type, "insufficient_data": True,
                 "n_train": len(train_df), "n_train_pos": n_train_pos}
            )
            continue

        feature_columns = get_non_constant_columns(feature_columns_all, train_df)
        model = build_model()
        model.fit(train_df[feature_columns], train_df[TARGET])

        val_proba = model.predict_proba(val_df[feature_columns])[:, 1]
        threshold = select_threshold(val_df[TARGET], val_proba)
        result = evaluate_at_threshold(val_df[TARGET], val_proba, threshold)
        result["variant"] = variant["name"]
        result["inspection_type"] = inspection_type
        result["insufficient_data"] = False
        result["n_train"] = len(train_df)
        result["n_train_pos"] = n_train_pos
        volume_gap_results.append(result)

volume_gap_df = pd.DataFrame(volume_gap_results).set_index(["inspection_type", "variant"]).sort_index()
cols = ["n_train", "n_train_pos", "threshold", "tn", "fp", "fn", "tp", "volume_reduction", "total_cost_1:10", "total_cost_1:100"]
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    print(f"=== type {inspection_type} ===")
    display(volume_gap_df.loc[inspection_type][[c for c in cols if c in volume_gap_df.columns]])


=== type 0 ===


,n_train,n_train_pos,threshold,tn,fp,fn,tp,volume_reduction,total_cost_1:10,total_cost_1:100
variant,,,,,,,,,,
size20_gap0,13409,21,6.546095e-07,2423.0,21061.0,0.0,8.0,0.103177,21061.0,21061.0
size20_gap20,11797,19,1.133124e-05,10993.0,12491.0,0.0,8.0,0.468106,12491.0,12491.0
size40_gap0,25206,40,4.212050e-06,10685.0,12799.0,0.0,8.0,0.454991,12799.0,12799.0
size40_gap20,28552,36,9.708172e-06,13207.0,10277.0,0.0,8.0,0.562383,10277.0,10277.0


=== type 1 ===


,n_train,n_train_pos,threshold,tn,fp,fn,tp,volume_reduction,total_cost_1:10,total_cost_1:100
variant,,,,,,,,,,
size20_gap0,7097,258,0.000078,4126.0,6129.0,2.0,237.0,0.402340,6149.0,6329.0
size20_gap20,11311,84,0.000008,2464.0,7791.0,2.0,237.0,0.240273,7811.0,7991.0
size40_gap0,18408,342,0.000034,2529.0,7726.0,2.0,237.0,0.246611,7746.0,7926.0
size40_gap20,26944,297,0.000004,629.0,9626.0,2.0,237.0,0.061336,9646.0,9826.0


=== type 2 ===


,n_train,n_train_pos,threshold,tn,fp,fn,tp,volume_reduction,total_cost_1:10,total_cost_1:100
variant,,,,,,,,,,
size20_gap0,22317,79,3.943738e-06,2540.0,13325.0,0.0,32.0,0.160101,13325.0,13325.0
size20_gap20,32899,210,5.307290e-07,184.0,15681.0,0.0,32.0,0.011598,15681.0,15681.0
size40_gap0,55216,289,1.574087e-06,700.0,15165.0,0.0,32.0,0.044122,15165.0,15165.0
size40_gap20,52593,500,3.950262e-06,657.0,15208.0,0.0,32.0,0.041412,15208.0,15208.0


=== type 3 ===


,n_train,n_train_pos,threshold,tn,fp,fn,tp,volume_reduction,total_cost_1:10,total_cost_1:100
variant,,,,,,,,,,
size20_gap0,34547,61,0.000003,9244.0,18060.0,0.0,24.0,0.338558,18060.0,18060.0
size20_gap20,21167,298,0.000020,6829.0,20475.0,0.0,24.0,0.250110,20475.0,20475.0
size40_gap0,55714,359,0.000005,7538.0,19766.0,0.0,24.0,0.276077,19766.0,19766.0
size40_gap20,46245,559,0.000014,12785.0,14519.0,0.0,24.0,0.468246,14519.0,14519.0


=== type 4 ===


,n_train,n_train_pos,threshold,tn,fp,fn,tp,volume_reduction,total_cost_1:10,total_cost_1:100
variant,,,,,,,,,,
size20_gap0,1052,5,0.000051,63.0,1040.0,0.0,60.0,0.057117,1040.0,1040.0
size20_gap20,1220,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
size40_gap0,2272,9,0.000004,6.0,1097.0,0.0,60.0,0.005440,1097.0,1097.0
size40_gap20,2466,8,0.000005,25.0,1078.0,0.0,60.0,0.022665,1078.0,1078.0


## 9. 결론 — "데이터가 적어서"가 아니라 유형마다 다른 이유로 갈린다

### TN/FN 비교 (Val=60~80% 고정, Train만 4가지, 총비용 1:10 기준)

| type | gap0·size40(20~60%) | gap20·size40(0~40%) | gap0·size20(40~60%) | gap20·size20(20~40%) |
|---|---|---|---|---|
| 0 | TN 10,685/FN 0 (12,799) | TN 13,207/FN 0 (**10,277**) | TN 2,423/FN 0 (21,061) | TN 10,993/FN 0 (12,491) |
| 1 | TN 2,529/FN 2 (7,746) | TN 629/FN 2 (9,646) | TN 4,126/FN 2 (**6,149**) | TN 2,464/FN 2 (7,811) |
| 2 | TN 700/FN 0 (15,165) | TN 657/FN 0 (15,208) | TN 2,540/FN 0 (**13,325**) | TN 184/FN 0 (15,681) |
| 3 | TN 7,538/FN 0 (19,766) | TN 12,785/FN 0 (**14,519**) | TN 9,244/FN 0 (18,060) | TN 6,829/FN 0 (20,475) |
| 4 | TN 6/FN 0 (1,097) | TN 25/FN 0 (1,078) | TN 63/FN 0 (**1,040**) | 표본 부족(양성 4건) |

(괄호는 총비용 1:10, 굵게 표시가 그 유형의 최고 조합. FN이 거의 다 0~2로 같아서 **TN 차이가
곧 결론**이다.)

### size(데이터 양) 효과 — 같은 gap에서 40% vs 20%

| type | gap=0에서 승자 | gap=20에서 승자 |
|---|---|---|
| 0 | size40(12,799) < size20(21,061) → **큰 게 이김** | size40(10,277) < size20(12,491) → **큰 게 이김** |
| 1 | size20(6,149) < size40(7,746) → **작은 게 이김** | size20(7,811) < size40(9,646) → **작은 게 이김** |
| 2 | size20(13,325) < size40(15,165) → **작은 게 이김** | size40(15,208) < size20(15,681) → 큰 게 이김(근소) |
| 3 | size20(18,060) < size40(19,766) → 작은 게 이김(근소) | size40(14,519) ≪ size20(20,475) → **큰 게 확실히 이김** |
| 4 | 표본 부족, 판단 보류 | 표본 부족, 판단 보류 |

**type0은 데이터 양이 일관되게 도움이 되고(항상 size40 승), type1은 정반대로 일관되게
데이터를 더 넣는 게 손해다.** type2/3은 gap에 따라 승자가 바뀐다 — size 효과가 gap과
독립적이지 않고 서로 얽혀 있다는 뜻.

### gap(최신성) 효과 — 같은 size에서 gap0 vs gap20

| type | size=40에서 승자 | size=20에서 승자 |
|---|---|---|
| 0 | gap20(10,277) < gap0(12,799) → **오래된 쪽이 이김** | gap20(12,491) ≪ gap0(21,061) → **오래된 쪽이 확실히 이김** |
| 1 | gap0(7,746) < gap20(9,646) → **최신이 이김** | gap0(6,149) < gap20(7,811) → **최신이 이김** |
| 2 | gap0(15,165) ≈ gap20(15,208) → 거의 동률(최신 근소 우위) | gap0(13,325) < gap20(15,681) → **최신이 이김** |
| 3 | gap20(14,519) ≪ gap0(19,766) → **오래된 쪽이 이김** | gap0(18,060) < gap20(20,475) → **최신이 이김** |
| 4 | gap20(1,078) ≈ gap0(1,097) → 거의 동률 | (size20_gap20 표본 부족) |

**type0은 size와 무관하게 항상 "오래된 데이터"가 이긴다** — 40~60% 구간 자체가 학습에 오히려
방해가 되는 뭔가 이례적인 특성을 가진 것으로 보인다(주간 집계에서 본 type0의 극단적 저표본
구간이 이 안에 있을 가능성). **type1은 size와 무관하게 항상 "최신 데이터"가 이긴다** — 가장
직관에 맞는 유형. **type3은 size에 따라 승자가 뒤집힌다**(size40에서는 오래된 쪽, size20에서는
최신 쪽) — size와 gap 효과가 상호작용한다는 뜻이라 한 문장으로 요약하기 어렵다.

### 종합 답변 (사용자 질문에 대한 직접 답)

**"재학습하면 성능이 주는 게 데이터가 적어져서 아니냐"는 질문의 답은 "유형마다 다르고, 어느
쪽도 100% 설명하지 못한다"** 입니다:

- **type0**: 데이터 양도 도움이 되고(size40>size20 항상), 그런데 최신 데이터는 오히려
  해롭다(gap20>gap0 항상). 5절의 "재학습해도 나빠진다"는 결론은 **데이터 양 감소 때문이
  아니라 진짜 drift(최신 구간 자체의 이례적 특성) 때문**이라는 게 이번 분리로 더 명확해졌다.
- **type1**: 정반대로 데이터를 더 넣으면 오히려 손해(size20이 항상 이김)고 최신성은 도움이
  된다. 008에서 "type1은 frozen(적게, 오래전 학습)이 재학습보다 낫다"고 봤던 것과 겉으로는
  모순돼 보이지만 — frozen은 gap이 점점 벌어지는 쪽이고 여기 gap20도 오래된 쪽이라 방향이
  다르다. 정확한 정합성 확인은 추가 검토가 필요.
- **type2/3**: size와 gap이 서로 얽혀서 방향이 바뀐다 — 단순한 "양이 문제"나 "최신성이 문제"
  중 하나로 환원되지 않는다.
- **핵심**: 사용자가 의심한 "단순 데이터 양 감소 효과"는 **type0/3(gap=20 조건)에서는 실제로
  존재**하지만, 그것과 **별개로 최신성 자체의 (종종 반직관적인) 효과도 확인**된다. 두 효과가
  유형마다 다른 크기·방향으로 섞여 있어서, 5절처럼 크기를 고정한 채 시간만 미는 실험 하나로는
  둘을 구분할 수 없었다는 게 이번 절의 방법론적 기여다.
